# QM9 Dataset Preview

This notebook extracts the local QM9 archive, parses the `.xyz` files, loads the molecular property records into pandas, and displays the first 10 molecules.

In [2]:
from pathlib import Path
import tarfile
from collections import Counter

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

ARCHIVE_PATH = Path("dsgdb9nsd.xyz.tar.bz2")
EXTRACT_DIR = Path("data/qm9_xyz")

ARCHIVE_PATH, EXTRACT_DIR

(PosixPath('dsgdb9nsd.xyz.tar.bz2'), PosixPath('data/qm9_xyz'))

## 1. Extract the Dataset

In [3]:
def safe_extract_tar(tar: tarfile.TarFile, destination: Path) -> None:
    """Extract tar members only inside destination."""
    destination = destination.resolve()
    for member in tar.getmembers():
        target = (destination / member.name).resolve()
        if not str(target).startswith(str(destination)):
            raise RuntimeError(f"Blocked unsafe tar member: {member.name}")
    tar.extractall(destination)


if not ARCHIVE_PATH.exists():
    raise FileNotFoundError(f"Archive not found: {ARCHIVE_PATH.resolve()}")

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

existing_xyz = sorted(EXTRACT_DIR.glob("*.xyz"))
if existing_xyz:
    print(f"Dataset already extracted: {len(existing_xyz):,} xyz files in {EXTRACT_DIR}")
else:
    print(f"Extracting {ARCHIVE_PATH} to {EXTRACT_DIR} ...")
    with tarfile.open(ARCHIVE_PATH, "r:bz2") as tar:
        safe_extract_tar(tar, EXTRACT_DIR)
    existing_xyz = sorted(EXTRACT_DIR.glob("*.xyz"))
    print(f"Extracted {len(existing_xyz):,} xyz files.")

Extracting dsgdb9nsd.xyz.tar.bz2 to data/qm9_xyz ...


/var/folders/kf/s_6smlsj14s1mgmtn0d49cfw0000gn/T/ipykernel_65678/1987838874.py:8: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(destination)


Extracted 133,885 xyz files.


## 2. Parse QM9 `.xyz` Files

Each QM9 file contains atom coordinates plus molecular properties. The second line stores the main target properties used in many machine learning tasks.

In [4]:
PROPERTY_COLUMNS = [
    "tag",
    "molecule_index",
    "A",
    "B",
    "C",
    "mu",
    "alpha",
    "homo",
    "lumo",
    "gap",
    "r2",
    "zpve",
    "U0",
    "U",
    "H",
    "G",
    "Cv",
]


def molecular_formula(symbols):
    counts = Counter(symbols)
    ordered = []
    for symbol in ["C", "H", "N", "O", "F"]:
        if symbol in counts:
            count = counts[symbol]
            ordered.append(symbol if count == 1 else f"{symbol}{count}")
    for symbol in sorted(set(counts) - {"C", "H", "N", "O", "F"}):
        count = counts[symbol]
        ordered.append(symbol if count == 1 else f"{symbol}{count}")
    return "".join(ordered)


def parse_qm9_xyz(path: Path) -> dict:
    lines = path.read_text().splitlines()
    num_atoms = int(lines[0].strip())
    property_values = lines[1].split()

    record = {
        "file": path.name,
        "num_atoms": num_atoms,
    }

    for key, value in zip(PROPERTY_COLUMNS, property_values):
        if key == "tag":
            record[key] = value
        elif key == "molecule_index":
            record[key] = int(value)
        else:
            record[key] = float(value)

    atom_lines = lines[2 : 2 + num_atoms]
    symbols = []
    coordinates = []
    mulliken_charges = []
    for atom_line in atom_lines:
        symbol, x, y, z, charge = atom_line.split()
        symbols.append(symbol)
        coordinates.append((float(x), float(y), float(z)))
        mulliken_charges.append(float(charge))

    frequency_index = 2 + num_atoms
    smiles_index = frequency_index + 1
    inchi_index = frequency_index + 2

    smiles_values = lines[smiles_index].split()
    inchi_values = lines[inchi_index].split()

    record.update(
        {
            "formula": molecular_formula(symbols),
            "atom_symbols": symbols,
            "coordinates_angstrom": coordinates,
            "mulliken_charges": mulliken_charges,
            "frequencies_cm_minus_1": [float(x) for x in lines[frequency_index].split()],
            "smiles_gdb": smiles_values[0] if len(smiles_values) > 0 else None,
            "smiles_relaxed": smiles_values[1] if len(smiles_values) > 1 else None,
            "inchi_gdb": inchi_values[0] if len(inchi_values) > 0 else None,
            "inchi_relaxed": inchi_values[1] if len(inchi_values) > 1 else None,
        }
    )
    return record

## 3. Load and Display 10 Records

In [5]:
xyz_files = sorted(EXTRACT_DIR.glob("*.xyz"))
if not xyz_files:
    raise RuntimeError(f"No .xyz files found in {EXTRACT_DIR.resolve()}")

records = [parse_qm9_xyz(path) for path in xyz_files[:10]]
qm9_preview = pd.DataFrame(records)

preview_columns = [
    "file",
    "molecule_index",
    "formula",
    "num_atoms",
    "smiles_gdb",
    "mu",
    "alpha",
    "homo",
    "lumo",
    "gap",
    "zpve",
    "U0",
    "U",
    "H",
    "G",
    "Cv",
]

qm9_preview[preview_columns]

,file,molecule_index,formula,num_atoms,smiles_gdb,mu,alpha,homo,lumo,gap,zpve,U0,U,H,G,Cv
0,dsgdb9nsd_000001.xyz,1,CH4,5,C,0.0000,13.21,-0.3877,0.1171,0.5048,0.044749,-40.478930,-40.476062,-40.475117,-40.498597,6.469
1,dsgdb9nsd_000002.xyz,2,H3N,4,N,1.6256,9.46,-0.2570,0.0829,0.3399,0.034358,-56.525887,-56.523026,-56.522082,-56.544961,6.316
2,dsgdb9nsd_000003.xyz,3,H2O,3,O,1.8511,6.31,-0.2928,0.0687,0.3615,0.021375,-76.404702,-76.401867,-76.400922,-76.422349,6.002
3,dsgdb9nsd_000004.xyz,4,C2H2,4,C#C,0.0000,16.28,-0.2845,0.0506,0.3351,0.026841,-77.308427,-77.305527,-77.304583,-77.327429,8.574
4,dsgdb9nsd_000005.xyz,5,CHN,3,C#N,2.8937,12.99,-0.3604,0.0191,0.3796,0.016601,-93.411888,-93.409370,-93.408425,-93.431246,6.278
5,dsgdb9nsd_000006.xyz,6,CH2O,4,C=O,2.1089,14.18,-0.2670,-0.0406,0.2263,0.026603,-114.483613,-114.480746,-114.479802,-114.505268,6.413
6,dsgdb9nsd_000007.xyz,7,C2H6,8,CC,0.0000,23.95,-0.3385,0.1041,0.4426,0.074542,-79.764152,-79.760666,-79.759722,-79.787269,10.098
7,dsgdb9nsd_000008.xyz,8,CH4O,6,CO,1.5258,16.97,-0.2653,0.0784,0.3437,0.051208,-115.679136,-115.675816,-115.674872,-115.701876,8.751
8,dsgdb9nsd_000009.xyz,9,C3H4,7,CC#C,0.7156,28.78,-0.2609,0.0613,0.3222,0.055410,-116.609549,-116.605550,-116.604606,-116.633775,12.482
9,dsgdb9nsd_000010.xyz,10,C2H3N,6,CC#N,3.8266,24.45,-0.3264,0.0376,0.3640,0.045286,-132.718150,-132.714563,-132.713619,-132.742149,10.287


In [ ]:
# The full parsed information for the same 10 molecules is also available here.
qm9_preview